# Drug–Disease Relation Extraction with Fine-Tuned BioBERT

Training and evaluation notebook for the B.Sc. Computer Science final-year project *A Deep Learning-Based Drug-Disease Relation Extraction Model Using scispaCy and Textual Data from Biomedical Literature* (Olujobi Damilola David, Covenant University, 2026).

Two relation types are modelled as separate binary classification tasks:

| Task | Corpus | Positive label |
|---|---|---|
| causes | BC5CDR (1,500 PubMed abstracts) | chemical-induced disease (CID) relation |
| treats | BioRED (600 PubMed abstracts) | chemical–disease `Negative_Correlation` (proxy for treats) |

For each task, seven models are compared on the same held-out test set: TF-IDF features with logistic regression, a linear SVM and LightGBM; frozen BioBERT embeddings with the same three classifiers; and BioBERT (`dmis-lab/biobert-base-cased-v1.1`) fine-tuned end to end. Examples are built at sentence level by distant supervision, with the chemical mention wrapped in `[E1] … [/E1]` and the disease mention in `[E2] … [/E2]`.

**Environment:** Google Colab with an NVIDIA Tesla T4 GPU, Transformers 4.49.0. Trained models are saved to Google Drive (`chapter4_models/`).

In [ ]:
!pip install -q "transformers==4.49.0" datasets accelerate scikit-learn nltk lightgbm
import torch
print("CUDA available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU detected")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
CUDA available: True - Tesla T4


### Model storage

Trained models are written to Google Drive so that they persist after the Colab session ends. Outside Colab, a local `chapter4_models/` folder is used instead.

In [ ]:
# --- Model saving setup ---
# Colab runtimes are wiped when the session ends, so models are saved to Google Drive.
# If Drive can't be mounted (e.g. running locally), it falls back to a local folder.
import os, joblib

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/chapter4_models"
except Exception as e:
    print("Drive not mounted, saving locally instead:", e)
    SAVE_DIR = "./chapter4_models"

os.makedirs(SAVE_DIR, exist_ok=True)
print("Models will be saved to:", SAVE_DIR)

def save_sklearn(obj, name):
    """Save an sklearn / LightGBM object (vectorizer or classifier) with joblib."""
    path = os.path.join(SAVE_DIR, name + ".joblib")
    joblib.dump(obj, path)
    print("saved", path)

def save_hf(trainer_obj, tok, name):
    """Save a fine-tuned HuggingFace model + tokenizer (incl. the [E1]/[E2] marker tokens)."""
    path = os.path.join(SAVE_DIR, name)
    trainer_obj.save_model(path)
    tok.save_pretrained(path)
    print("saved", path)

## 1. BC5CDR corpus (causes task)

The official BioCreative V CDR release is downloaded in PubTator format and parsed into title, abstract, entity mentions with their MeSH identifiers, and document-level CID relations, for the training, development and test sets (500 abstracts each). The files are read directly rather than through the Hugging Face `datasets` hub, whose BC5CDR loader relies on a loading script that current `datasets` versions no longer run.

In [ ]:
!wget -q https://raw.githubusercontent.com/JHnlp/BioCreative-V-CDR-Corpus/master/CDR_Data.zip -O CDR_Data.zip
!unzip -oq CDR_Data.zip -d cdr_data

import glob
pubtator_files = sorted(glob.glob("cdr_data/**/*.PubTator.txt", recursive=True))
for f in pubtator_files:
    print(f)

cdr_data/CDR_Data/CDR.Corpus.v010516/CDR_DevelopmentSet.PubTator.txt
cdr_data/CDR_Data/CDR.Corpus.v010516/CDR_TestSet.PubTator.txt
cdr_data/CDR_Data/CDR.Corpus.v010516/CDR_TrainingSet.PubTator.txt
cdr_data/CDR_Data/DNorm.TestSet/TestSet.DNorm.PubTator.txt
cdr_data/CDR_Data/tmChem.TestSet/TestSet.tmChem.PubTator.txt


In [ ]:
import re

def parse_pubtator(path):
    """Parse a CDR PubTator file into {pmid: {title, abstract, entities, relations}}."""
    with open(path, encoding="utf-8") as f:
        lines = f.read().strip("\n").split("\n")

    docs = {}
    i = 0
    n = len(lines)
    while i < n:
        line = lines[i]
        if not line.strip():
            i += 1
            continue

        m_t = re.match(r"^(\d+)\|t\|(.*)$", line)
        pmid, title = m_t.group(1), m_t.group(2)
        i += 1

        m_a = re.match(r"^(\d+)\|a\|(.*)$", lines[i])
        abstract = m_a.group(2)
        i += 1

        entities, relations = [], []
        while i < n and lines[i].strip():
            parts = lines[i].split("\t")
            if parts[1] == "CID":
                # PMID  CID  chemical_mesh_id  disease_mesh_id
                relations.append({"chemical_id": parts[2], "disease_id": parts[3]})
            else:
                # PMID  start  end  text  type  mesh_id
                entities.append({
                    "start": int(parts[1]),
                    "end": int(parts[2]),
                    "text": parts[3],
                    "type": parts[4],
                    "mesh_id": parts[5],
                })
            i += 1

        docs[pmid] = {"title": title, "abstract": abstract, "entities": entities, "relations": relations}
        i += 1  # skip the blank line separating documents

    return docs

path_map = {}
for f in pubtator_files:
    if "TrainingSet" in f:
        path_map["train"] = f
    elif "DevelopmentSet" in f:
        path_map["validation"] = f
    elif "TestSet" in f and "DNorm" not in f and "tmChem" not in f:
        path_map["test"] = f

assert set(path_map) == {"train", "validation", "test"}, f"Missing splits, found: {path_map}"

train_docs = parse_pubtator(path_map["train"])
val_docs   = parse_pubtator(path_map["validation"])
test_docs  = parse_pubtator(path_map["test"])

print("train docs:", len(train_docs))
print("val docs:  ", len(val_docs))
print("test docs: ", len(test_docs))

sample_pmid = next(iter(train_docs))
sample = train_docs[sample_pmid]
print()
print("Sample PMID:", sample_pmid)
print("Entities:", sample["entities"][:3])
print("Relations:", sample["relations"][:3])


train docs: 500
val docs:   500
test docs:  500

Sample PMID: 227508
Entities: [{'start': 0, 'end': 8, 'text': 'Naloxone', 'type': 'Chemical', 'mesh_id': 'D009270'}, {'start': 49, 'end': 58, 'text': 'clonidine', 'type': 'Chemical', 'mesh_id': 'D003000'}, {'start': 93, 'end': 105, 'text': 'hypertensive', 'type': 'Disease', 'mesh_id': 'D006973'}]
Relations: [{'chemical_id': 'D008750', 'disease_id': 'D007022'}]


## 2. Sentence-level examples (causes task)

A (chemical, disease) pair becomes a candidate example only when both mentions occur in the same sentence; sentences are found with NLTK's Punkt tokenizer using character offsets. The label comes from the document-level gold CID relations: 1 if the MeSH-identifier pair is a gold relation of the abstract, 0 otherwise. Mentions without a MeSH identifier are skipped. In each example the chemical mention is wrapped in `[E1] … [/E1]` and the disease mention in `[E2] … [/E2]`.

In [ ]:
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import PunktSentenceTokenizer

SENT_TOKENIZER = PunktSentenceTokenizer()

def build_examples(docs):
    """
    Sentence-level distant supervision (Chapter 3 design): a (chemical, disease) MeSH-ID
    pair is only turned into a candidate example if both entities occur in the SAME sentence.
    Ground truth still comes from the document's gold CID pairs (BC5CDR has no sentence-level
    gold labels), but requiring same-sentence co-occurrence is a much stronger evidence condition
    than abstract-level co-occurrence. Each example wraps the chemical mention in [E1]/[/E1] and
    the disease mention in [E2]/[/E2] marker tokens, per Section 3.4.2.
    Entities with an unnormalized MeSH ID ('-1') are skipped since relations can't reference them.
    """
    examples = []
    for pmid, doc in docs.items():
        full_text = doc["title"] + " " + doc["abstract"]
        gold_pairs = {(r["chemical_id"], r["disease_id"]) for r in doc["relations"]}

        for s_start, s_end in SENT_TOKENIZER.span_tokenize(full_text):
            sent_ents = [e for e in doc["entities"]
                         if e["mesh_id"] != "-1" and e["start"] >= s_start and e["end"] <= s_end]
            chems = [e for e in sent_ents if e["type"] == "Chemical"]
            diseases = [e for e in sent_ents if e["type"] == "Disease"]
            if not chems or not diseases:
                continue

            for c in chems:
                for d in diseases:
                    if c["mesh_id"] == d["mesh_id"]:
                        continue
                    label = 1 if (c["mesh_id"], d["mesh_id"]) in gold_pairs else 0

                    # insert [E1]/[E2] marker tokens around each mention, in text order
                    spans = sorted([(c["start"], c["end"], "[E1]", "[/E1]"),
                                     (d["start"], d["end"], "[E2]", "[/E2]")])
                    marked, cursor = "", s_start
                    for st, en, open_tag, close_tag in spans:
                        marked += full_text[cursor:st] + f"{open_tag} {full_text[st:en]} {close_tag}"
                        cursor = en
                    marked += full_text[cursor:s_end]

                    examples.append({
                        "text": marked.strip(),
                        "label": label,
                        "pmid": pmid,
                        "chemical_id": c["mesh_id"],
                        "disease_id": d["mesh_id"],
                    })
    return examples

train_examples = build_examples(train_docs)
val_examples   = build_examples(val_docs)
test_examples  = build_examples(test_docs)

print("train:", len(train_examples), "pos:", sum(e["label"] for e in train_examples))
print("val:  ", len(val_examples),   "pos:", sum(e["label"] for e in val_examples))
print("test: ", len(test_examples),  "pos:", sum(e["label"] for e in test_examples))


train: 4562 pos: 1710
val:   4961 pos: 1858
test:  4888 pos: 1844


The training set is capped at 3,000 examples to keep the cost of training seven models manageable: all 1,710 positive examples are kept and 1,290 negative examples are sampled (seed 42). The validation and test sets are used in full.

In [ ]:
import random
random.seed(42)

MAX_TRAIN = 3000   # set to None to use the full training set
if MAX_TRAIN is not None and len(train_examples) > MAX_TRAIN:
    pos = [e for e in train_examples if e["label"] == 1]
    neg = [e for e in train_examples if e["label"] == 0]
    random.shuffle(neg)
    neg = neg[: MAX_TRAIN - len(pos)]
    train_examples = pos + neg
    random.shuffle(train_examples)

print("Using", len(train_examples), "training examples")


Using 3000 training examples


## 3. Baseline 1: TF-IDF features (causes task)

### 3a. TF-IDF + logistic regression

A TF-IDF vectoriser (unigrams and bigrams, 20,000 features) is fitted on the training examples and paired with a class-weighted logistic regression.

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

X_train = [e["text"] for e in train_examples]
y_train = [e["label"] for e in train_examples]
X_test  = [e["text"] for e in test_examples]
y_test  = [e["label"] for e in test_examples]

vec = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(Xtr, y_train)
preds = clf.predict(Xte)

print("=== BASELINE (TF-IDF + LogReg) ===")
print(classification_report(y_test, preds, digits=3))
print(confusion_matrix(y_test, preds))


=== BASELINE (TF-IDF + LogReg) ===
              precision    recall  f1-score   support

           0      0.794     0.624     0.699      3044
           1      0.542     0.733     0.623      1844

    accuracy                          0.665      4888
   macro avg      0.668     0.679     0.661      4888
weighted avg      0.699     0.665     0.670      4888

[[1900 1144]
 [ 492 1352]]


### 3b. TF-IDF + linear SVM and TF-IDF + LightGBM

The same TF-IDF features are paired with a class-weighted linear SVM and with LightGBM, so that differences between the three TF-IDF models reflect the classifier alone.

In [ ]:
from sklearn.svm import LinearSVC

clf_svm = LinearSVC(class_weight="balanced", max_iter=5000)
clf_svm.fit(Xtr, y_train)
preds_svm = clf_svm.predict(Xte)

print("=== BASELINE (TF-IDF + SVM) ===")
print(classification_report(y_test, preds_svm, digits=3))
print(confusion_matrix(y_test, preds_svm))


=== BASELINE (TF-IDF + SVM) ===
              precision    recall  f1-score   support

           0      0.809     0.617     0.700      3044
           1      0.546     0.760     0.635      1844

    accuracy                          0.671      4888
   macro avg      0.678     0.688     0.668      4888
weighted avg      0.710     0.671     0.676      4888

[[1877 1167]
 [ 442 1402]]


In [ ]:
import lightgbm as lgb

clf_lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    is_unbalance=True,
    random_state=42,
)
clf_lgbm.fit(Xtr, y_train)
preds_lgbm = clf_lgbm.predict(Xte)

print("=== BASELINE (TF-IDF + LightGBM) ===")
print(classification_report(y_test, preds_lgbm, digits=3))
print(confusion_matrix(y_test, preds_lgbm))


[LightGBM] [Info] Number of positive: 1710, number of negative: 1290
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.042285 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 22522
[LightGBM] [Info] Number of data points in the train set: 3000, number of used features: 1113
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.570000 -> initscore=0.281851
[LightGBM] [Info] Start training from score 0.281851


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


=== BASELINE (TF-IDF + LightGBM) ===
              precision    recall  f1-score   support

           0      0.767     0.534     0.629      3044
           1      0.488     0.732     0.585      1844

    accuracy                          0.609      4888
   macro avg      0.627     0.633     0.607      4888
weighted avg      0.661     0.609     0.613      4888

[[1625 1419]
 [ 494 1350]]


In [ ]:
# Save causes-task TF-IDF baselines (vectoriser + 3 classifiers)
save_sklearn(vec,      "causes_tfidf_vectorizer")
save_sklearn(clf,      "causes_tfidf_logreg")
save_sklearn(clf_svm,  "causes_tfidf_svm")
save_sklearn(clf_lgbm, "causes_tfidf_lgbm")

In [ ]:
!pip install sentencepiece
!pip install tiktoken

### 3c. Baseline 2: frozen BioBERT embeddings (causes task)

BioBERT is used as a fixed feature extractor; its weights are not updated. Each marked sentence is mean-pooled over its tokens (padding excluded through the attention mask) into one 768-dimensional vector. The embeddings are computed once and paired with the same three classifiers (2a logistic regression, 2b linear SVM, 2c LightGBM), so that a difference from Baseline 1 at a given classifier reflects the representation. The marker strings stay plain text here, because new token embeddings would remain untrained without fine-tuning.

In [ ]:
import numpy as np
from transformers import AutoModel, AutoTokenizer

FROZEN_MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

frozen_tokenizer = AutoTokenizer.from_pretrained(FROZEN_MODEL_NAME, use_fast=False)
frozen_model = AutoModel.from_pretrained(FROZEN_MODEL_NAME).to(device)
frozen_model.eval()

def extract_frozen_embeddings(texts, batch_size=32, max_length=256):
    all_vecs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = frozen_tokenizer(batch, truncation=True, padding=True,
                                    max_length=max_length, return_tensors="pt").to(device)
            out = frozen_model(**enc)
            hidden = out.last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1)
            summed = (hidden * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            mean_pooled = summed / counts
            all_vecs.append(mean_pooled.cpu().numpy())
    return np.concatenate(all_vecs, axis=0)

# Extracted once here, reused by 2a/2b/2c below (mirrors Xtr/Xte reuse in Baseline 1)
X_train_frozen = extract_frozen_embeddings(X_train)
X_test_frozen = extract_frozen_embeddings(X_test)
print("Frozen embedding shape - train:", X_train_frozen.shape, "test:", X_test_frozen.shape)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Frozen embedding shape - train: (3000, 768) test: (4888, 768)


#### 2a. Frozen BioBERT + logistic regression

In [ ]:
clf_frozen_logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_frozen_logreg.fit(X_train_frozen, y_train)
preds_frozen_logreg = clf_frozen_logreg.predict(X_test_frozen)

print("=== BASELINE 2a (Frozen BioBERT + Logistic Regression) ===")
print(classification_report(y_test, preds_frozen_logreg, digits=3))
print(confusion_matrix(y_test, preds_frozen_logreg))


=== BASELINE 2a (Frozen BioBERT + Logistic Regression) ===
              precision    recall  f1-score   support

           0      0.787     0.654     0.715      3044
           1      0.554     0.708     0.622      1844

    accuracy                          0.675      4888
   macro avg      0.671     0.681     0.668      4888
weighted avg      0.699     0.675     0.680      4888

[[1992 1052]
 [ 538 1306]]


#### 2b. Frozen BioBERT + linear SVM

In [ ]:
clf_frozen_svm = LinearSVC(class_weight="balanced", max_iter=5000)
clf_frozen_svm.fit(X_train_frozen, y_train)
preds_frozen_svm = clf_frozen_svm.predict(X_test_frozen)

print("=== BASELINE 2b (Frozen BioBERT + SVM) ===")
print(classification_report(y_test, preds_frozen_svm, digits=3))
print(confusion_matrix(y_test, preds_frozen_svm))


=== BASELINE 2b (Frozen BioBERT + SVM) ===
              precision    recall  f1-score   support

           0      0.774     0.634     0.697      3044
           1      0.535     0.695     0.604      1844

    accuracy                          0.657      4888
   macro avg      0.655     0.664     0.651      4888
weighted avg      0.684     0.657     0.662      4888

[[1930 1114]
 [ 563 1281]]


#### 2c. Frozen BioBERT + LightGBM

In [ ]:
clf_frozen_lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    is_unbalance=True,
    random_state=42,
)
clf_frozen_lgbm.fit(X_train_frozen, y_train)
preds_frozen_lgbm = clf_frozen_lgbm.predict(X_test_frozen)

print("=== BASELINE 2c (Frozen BioBERT + LightGBM) ===")
print(classification_report(y_test, preds_frozen_lgbm, digits=3))
print(confusion_matrix(y_test, preds_frozen_lgbm))


[LightGBM] [Info] Number of positive: 1710, number of negative: 1290
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.187080 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 3000, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.570000 -> initscore=0.281851
[LightGBM] [Info] Start training from score 0.281851
=== BASELINE 2c (Frozen BioBERT + LightGBM) ===
              precision    recall  f1-score   support

           0      0.792     0.583     0.672      3044
           1      0.521     0.747     0.614      1844

    accuracy                          0.645      4888
   macro avg      0.656     0.665     0.643      4888
weighted avg      0.690     0.645     0.650      4888

[[1776 1268]
 [ 466 1378]]


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# Save causes-task frozen-BioBERT classifiers
# (the frozen encoder itself is just the public dmis-lab/biobert-base-cased-v1.1, so it isn't re-saved)
save_sklearn(clf_frozen_logreg, "causes_frozen_logreg")
save_sklearn(clf_frozen_svm,    "causes_frozen_svm")
save_sklearn(clf_frozen_lgbm,   "causes_frozen_lgbm")

## 4. Fine-tuned BioBERT (causes task)

`dmis-lab/biobert-base-cased-v1.1` with a two-class sequence-classification head is fine-tuned end to end. The four marker tokens are added to the tokenizer as special tokens, so they are never split into word pieces, and the embedding matrix is resized to match. The loss is class-weighted cross-entropy, with the positive-class weight set to n_neg / n_pos on the training set.

Settings: 3 epochs, learning rate 2 × 10⁻⁵, batch size 16, maximum length 256 tokens, mixed-precision training. Validation metrics are computed after every epoch, and the final (epoch-3) model is kept.

In [ ]:

from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, DataCollatorWithPadding)
import numpy as np
from datasets import Dataset

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]})

def to_hf_dataset(examples):
    return Dataset.from_dict({
        "text": [e["text"] for e in examples],
        "label": [e["label"] for e in examples],
    })

train_ds = to_hf_dataset(train_examples)
val_ds   = to_hf_dataset(val_examples)
test_ds  = to_hf_dataset(test_examples)

def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = train_ds.map(tok, batched=True)
val_ds   = val_ds.map(tok, batched=True)
test_ds  = test_ds.map(tok, batched=True)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

# class weights for the loss: positive-class weight = n_neg / n_pos (Equation 3.9)
n_pos = sum(y_train)
n_neg = len(y_train) - n_pos
class_weights = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float)
print("class weights [neg, pos]:", class_weights.tolist())

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.resize_token_embeddings(len(tokenizer))  # account for the added [E1]/[E2] marker tokens

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="./biobert_causes",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collator, compute_metrics=compute_metrics,
)

trainer.train()


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4961 [00:00<?, ? examples/s]

Map:   0%|          | 0/4888 [00:00<?, ? examples/s]

class weights [neg, pos]: [1.0, 0.7543859481811523]


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.536000,0.560347,0.711953,0.581590,0.822928,0.681524
2,0.365400,0.595840,0.736343,0.605283,0.850915,0.707383
3,0.221600,0.706459,0.749244,0.629318,0.804090,0.706049


TrainOutput(global_step=564, training_loss=0.38794239319808094, metrics={'train_runtime': 90.5349, 'train_samples_per_second': 99.409, 'train_steps_per_second': 6.23, 'total_flos': 519315445516800.0, 'train_loss': 0.38794239319808094, 'epoch': 3.0})

In [ ]:
# Save the fine-tuned causes-task BioBERT (weights + config + tokenizer)
save_hf(trainer, tokenizer, "biobert_causes_final")

## 5. Test-set evaluation (causes task)

Precision, recall and F1 are printed for both classes; the positive class (CID relation) is the one reported in the project report.

In [ ]:

test_results = trainer.evaluate(test_ds)
print("=== BioBERT test results ===")
for k, v in test_results.items():
    print(f"{k}: {v}")

preds_output = trainer.predict(test_ds)
preds = np.argmax(preds_output.predictions, axis=-1)
print()
print(classification_report(y_test, preds, digits=3))
print(confusion_matrix(y_test, preds))


=== BioBERT test results ===
eval_loss: 0.6983246207237244
eval_accuracy: 0.7461129296235679
eval_precision: 0.6282433007230965
eval_recall: 0.8009761388286334
eval_f1: 0.7041716328963051
eval_runtime: 6.5348
eval_samples_per_second: 747.991
eval_steps_per_second: 23.413
epoch: 3.0

              precision    recall  f1-score   support

           0      0.855     0.713     0.778      3044
           1      0.628     0.801     0.704      1844

    accuracy                          0.746      4888
   macro avg      0.742     0.757     0.741      4888
weighted avg      0.770     0.746     0.750      4888

[[2170  874]
 [ 367 1477]]


### Causes task: test-set results

| Model | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| TF-IDF + logistic regression | 0.665 | 0.542 | 0.733 | 0.623 |
| TF-IDF + linear SVM | 0.671 | 0.546 | 0.760 | 0.635 |
| TF-IDF + LightGBM | 0.609 | 0.488 | 0.732 | 0.585 |
| Frozen BioBERT + logistic regression | 0.675 | 0.554 | 0.708 | 0.622 |
| Frozen BioBERT + linear SVM | 0.657 | 0.535 | 0.695 | 0.604 |
| Frozen BioBERT + LightGBM | 0.645 | 0.521 | 0.747 | 0.614 |
| **Fine-tuned BioBERT** | **0.746** | **0.628** | **0.801** | **0.704** |

Precision, recall and F1 are for the positive (CID relation) class, on 4,888 test examples of which 1,844 are positive.

## 6. BioRED corpus (treats task)

BioRED (Luo et al., 2022) is downloaded from NCBI and its BioC-JSON training, development and test files are parsed. BioRED annotates both chemical and disease mentions with character offsets, so the treats task uses the same dual entity-marker input as the causes task.

BioRED has no explicit TREATS relation. In its annotation scheme a chemical that treats a disease is annotated as `Negative_Correlation` (for example, *bortezomib* and *multiple myeloma*), so a chemical–disease `Negative_Correlation` relation is used as a proxy for treats; every other relation type, and every unannotated pair, is negative. The proxy also covers other cases in which a chemical reduces a disease or symptom, so the treats results are results on this proxy task.

In [ ]:
!wget -q https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/BIORED.zip -O BIORED.zip
!unzip -oq BIORED.zip -d biored_data

import glob
print("All files found:")
for f in sorted(glob.glob("biored_data/**/*", recursive=True)):
    print(" ", f)

All files found:
  biored_data/BioRED
  biored_data/BioRED/Dev.BioC.JSON
  biored_data/BioRED/Dev.BioC.XML
  biored_data/BioRED/Dev.PubTator
  biored_data/BioRED/Test.BioC.JSON
  biored_data/BioRED/Test.BioC.XML
  biored_data/BioRED/Test.PubTator
  biored_data/BioRED/Train.BioC.JSON
  biored_data/BioRED/Train.BioC.XML
  biored_data/BioRED/Train.PubTator


The split files are located by name; only the BioC-JSON versions are used.

In [ ]:
import glob

def find_split_file(keyword):
    # BioRED ships each split as .BioC.JSON, .BioC.XML and .PubTator; only the JSON
    # files are parsed.
    candidates = glob.glob(f"biored_data/**/*{keyword}*.BioC.JSON", recursive=True)
    if not candidates:
        # looser match in case the naming differs; still JSON only
        candidates = [
            c for c in glob.glob(f"biored_data/**/*{keyword}*", recursive=True)
            if c.lower().endswith(".json")
        ]
    assert candidates, f"No .BioC.JSON file found matching *{keyword}* -- check the file listing above"
    return candidates[0]

biored_path_map = {
    "train": find_split_file("Train"),
    "validation": find_split_file("Dev"),
    "test": find_split_file("Test"),
}
print(biored_path_map)

{'train': 'biored_data/BioRED/Train.BioC.JSON', 'validation': 'biored_data/BioRED/Dev.BioC.JSON', 'test': 'biored_data/BioRED/Test.BioC.JSON'}


In [ ]:
import json
CHEMICAL_TYPES = {"chemicalentity", "chemical"}
DISEASE_TYPES  = {"diseaseorphenotypicfeature", "disease"}

def parse_biored_pubtator(path):
    """
    Parse a BioRED BioC-JSON file into {pmid: {full_text, entities, relations}}.

    BioC-JSON schema:
      data["documents"] -> list of {id, passages, relations}
      passage -> {offset, text, annotations}
      annotation -> {infons: {identifier, type}, text, locations: [{offset, length}, ...]}
      doc-level relation -> {infons: {entity1, entity2, type, novel}}
      Relations reference entities by infons["identifier"] (the concept ID), NOT by the
      annotation's own "id" field -- entity1/entity2 are matched against concept_id below.
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    docs = {}
    for doc in data["documents"]:
        pmid = doc["id"]

        # Rebuild the document text using each passage's own offset, so the
        # (document-absolute) annotation offsets line up correctly regardless of the
        # exact separator between passages.
        end = max(p["offset"] + len(p["text"]) for p in doc["passages"])
        buf = [" "] * end
        for p in doc["passages"]:
            buf[p["offset"]: p["offset"] + len(p["text"])] = p["text"]
        full_text = "".join(buf)

        entities = []
        for p in doc["passages"]:
            for ann in p["annotations"]:
                identifier = ann["infons"].get("identifier")
                etype = ann["infons"].get("type")
                if identifier is None or etype is None:
                    continue
                for loc in ann["locations"]:
                    entities.append({
                        "start": loc["offset"],
                        "end": loc["offset"] + loc["length"],
                        "text": ann["text"],
                        "type": etype,
                        "concept_id": identifier,
                    })

        relations = []
        for rel in doc.get("relations", []):
            infons = rel["infons"]
            relations.append({
                "relation_type": infons.get("type"),
                "entity1_id": infons.get("entity1"),
                "entity2_id": infons.get("entity2"),
                "novel": infons.get("novel"),
            })

        docs[pmid] = {"full_text": full_text, "entities": entities, "relations": relations}

    return docs

biored_train_docs = parse_biored_pubtator(biored_path_map["train"])
biored_val_docs   = parse_biored_pubtator(biored_path_map["validation"])
biored_test_docs  = parse_biored_pubtator(biored_path_map["test"])

print("train docs:", len(biored_train_docs))
print("val docs:  ", len(biored_val_docs))
print("test docs: ", len(biored_test_docs))

sample_pmid = next(iter(biored_train_docs))
sample = biored_train_docs[sample_pmid]
print()
print("Sample PMID:", sample_pmid)
print("Entity types seen (sanity-check these against CHEMICAL_TYPES/DISEASE_TYPES above):")
print(sorted({e["type"] for e in sample["entities"]}))
print("Sample entities:", sample["entities"][:4])
print("Sample relations:", sample["relations"][:4])
print("Relation types seen in this doc:", sorted({r["relation_type"] for r in sample["relations"]}))

train docs: 400
val docs:   100
test docs:  100

Sample PMID: 10491763
Entity types seen (sanity-check these against CHEMICAL_TYPES/DISEASE_TYPES above):
['ChemicalEntity', 'DiseaseOrPhenotypicFeature', 'GeneOrGeneProduct', 'OrganismTaxon', 'SequenceVariant']
Sample entities: [{'start': 0, 'end': 27, 'text': 'Hepatocyte nuclear factor-6', 'type': 'GeneOrGeneProduct', 'concept_id': '3175'}, {'start': 74, 'end': 90, 'text': 'type II diabetes', 'type': 'DiseaseOrPhenotypicFeature', 'concept_id': 'D003924'}, {'start': 140, 'end': 147, 'text': 'insulin', 'type': 'GeneOrGeneProduct', 'concept_id': '3630'}, {'start': 184, 'end': 217, 'text': 'hepatocyte nuclear factor (HNF)-6', 'type': 'GeneOrGeneProduct', 'concept_id': '3175'}]
Sample relations: [{'relation_type': 'Association', 'entity1_id': '3175', 'entity2_id': 'D003924', 'novel': 'No'}, {'relation_type': 'Positive_Correlation', 'entity1_id': 'D005947', 'entity2_id': '3630', 'novel': 'No'}, {'relation_type': 'Association', 'entity1_id': '

The printed entity types confirm that BioRED's chemical and disease types (`ChemicalEntity`, `DiseaseOrPhenotypicFeature`) match the type sets used to build the examples.

## 7. Sentence-level examples (treats task)

The procedure of Section 2 is applied to BioRED: a (chemical, disease) concept pair becomes a candidate example when both mentions occur in the same sentence, and its label is 1 if the abstract has a `Negative_Correlation` relation between the two concepts (in either direction) and 0 otherwise.

In [ ]:
def normalize_type(t):
    return t.strip().lower()

def build_treats_examples_biored(docs):
    """
    Sentence-level distant supervision for the treats task, BioRED version. Mirrors
    build_examples() from Section 2 almost exactly: a (chemical, disease) concept-ID pair
    is a candidate example only if both mentions occur in the SAME sentence. label = 1 iff
    the pair has a gold Negative_Correlation relation (treats proxy, see Section 7), else 0.
    Both entities get marker-wrapped ([E1] chemical, [E2] disease), matching the causes
    task's input format and Algorithm 2's dual-marker expectation.
    """
    examples = []
    for pmid, doc in docs.items():
        full_text = doc["full_text"]  # already assembled in Section 7's parser
        treats_pairs = {
            (r["entity1_id"], r["entity2_id"])
            for r in doc["relations"] if r["relation_type"] == "Negative_Correlation"
        } | {
            (r["entity2_id"], r["entity1_id"])
            for r in doc["relations"] if r["relation_type"] == "Negative_Correlation"
        }

        # same Punkt sentence splitter as Section 2
        for s_start, s_end in SENT_TOKENIZER.span_tokenize(full_text):
            sent_ents = [e for e in doc["entities"] if e["start"] >= s_start and e["end"] <= s_end]
            chems    = [e for e in sent_ents if normalize_type(e["type"]) in CHEMICAL_TYPES]
            diseases = [e for e in sent_ents if normalize_type(e["type"]) in DISEASE_TYPES]
            if not chems or not diseases:
                continue

            for c in chems:
                for d in diseases:
                    if c["concept_id"] == d["concept_id"]:
                        continue
                    label = 1 if (c["concept_id"], d["concept_id"]) in treats_pairs else 0

                    spans = sorted([(c["start"], c["end"], "[E1]", "[/E1]"),
                                     (d["start"], d["end"], "[E2]", "[/E2]")])
                    marked, cursor = "", s_start
                    for st, en, open_tag, close_tag in spans:
                        marked += full_text[cursor:st] + f"{open_tag} {full_text[st:en]} {close_tag}"
                        cursor = en
                    marked += full_text[cursor:s_end]

                    examples.append({
                        "text": marked.strip(),
                        "label": label,
                        "pmid": pmid,
                        "chemical_id": c["concept_id"],
                        "disease_id": d["concept_id"],
                    })
    return examples

treats_train_examples = build_treats_examples_biored(biored_train_docs)
treats_val_examples   = build_treats_examples_biored(biored_val_docs)
treats_test_examples  = build_treats_examples_biored(biored_test_docs)

print("treats train:", len(treats_train_examples), "pos:", sum(e["label"] for e in treats_train_examples))
print("treats val:  ", len(treats_val_examples),   "pos:", sum(e["label"] for e in treats_val_examples))
print("treats test: ", len(treats_test_examples),  "pos:", sum(e["label"] for e in treats_test_examples))

treats train: 2259 pos: 566
treats val:   666 pos: 145
treats test:  505 pos: 90


The same 3,000-example cap is applied, but it has no effect here: the treats training set has 2,259 examples, all of which are used.

In [ ]:
random.seed(42)
MAX_TRAIN_TREATS = 3000   # set to None to use the full training set

if MAX_TRAIN_TREATS is not None and len(treats_train_examples) > MAX_TRAIN_TREATS:
    pos = [e for e in treats_train_examples if e["label"] == 1]
    neg = [e for e in treats_train_examples if e["label"] == 0]
    random.shuffle(neg)
    neg = neg[: MAX_TRAIN_TREATS - len(pos)]
    treats_train_examples = pos + neg
    random.shuffle(treats_train_examples)

print("Using", len(treats_train_examples), "treats training examples")

Using 2259 treats training examples


## 8. Baseline 1: TF-IDF features (treats task)

The three classifiers of Section 3, with a TF-IDF vectoriser fitted on the treats training examples.

In [ ]:
X_train_treats = [e["text"] for e in treats_train_examples]
y_train_treats = [e["label"] for e in treats_train_examples]
X_test_treats  = [e["text"] for e in treats_test_examples]
y_test_treats  = [e["label"] for e in treats_test_examples]

vec_treats = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
Xtr_treats = vec_treats.fit_transform(X_train_treats)
Xte_treats = vec_treats.transform(X_test_treats)

clf_treats_logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_treats_logreg.fit(Xtr_treats, y_train_treats)
preds_treats_logreg = clf_treats_logreg.predict(Xte_treats)

print("=== TREATS BASELINE (TF-IDF + LogReg) ===")
print(classification_report(y_test_treats, preds_treats_logreg, digits=3))
print(confusion_matrix(y_test_treats, preds_treats_logreg))


=== TREATS BASELINE (TF-IDF + LogReg) ===
              precision    recall  f1-score   support

           0      0.865     0.896     0.880       415
           1      0.427     0.356     0.388        90

    accuracy                          0.800       505
   macro avg      0.646     0.626     0.634       505
weighted avg      0.787     0.800     0.793       505

[[372  43]
 [ 58  32]]


In [ ]:
clf_treats_svm = LinearSVC(class_weight="balanced", max_iter=5000)
clf_treats_svm.fit(Xtr_treats, y_train_treats)
preds_treats_svm = clf_treats_svm.predict(Xte_treats)

print("=== TREATS BASELINE (TF-IDF + SVM) ===")
print(classification_report(y_test_treats, preds_treats_svm, digits=3))
print(confusion_matrix(y_test_treats, preds_treats_svm))


=== TREATS BASELINE (TF-IDF + SVM) ===
              precision    recall  f1-score   support

           0      0.873     0.911     0.892       415
           1      0.486     0.389     0.432        90

    accuracy                          0.818       505
   macro avg      0.680     0.650     0.662       505
weighted avg      0.804     0.818     0.810       505

[[378  37]
 [ 55  35]]


In [ ]:
clf_treats_lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    is_unbalance=True,
    random_state=42,
)
clf_treats_lgbm.fit(Xtr_treats, y_train_treats)
preds_treats_lgbm = clf_treats_lgbm.predict(Xte_treats)

print("=== TREATS BASELINE (TF-IDF + LightGBM) ===")
print(classification_report(y_test_treats, preds_treats_lgbm, digits=3))
print(confusion_matrix(y_test_treats, preds_treats_lgbm))


[LightGBM] [Info] Number of positive: 566, number of negative: 1693
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012591 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 17609
[LightGBM] [Info] Number of data points in the train set: 2259, number of used features: 943
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.250553 -> initscore=-1.095663
[LightGBM] [Info] Start training from score -1.095663
=== TREATS BASELINE (TF-IDF + LightGBM) ===
              precision    recall  f1-score   support

           0      0.859     0.853     0.856       415
           1      0.344     0.356     0.350        90

    accuracy                          0.764       505
   macro avg      0.602     0.604     0.603       505
weighted avg      0.767     0.764     0.766       505

[[354  61]
 [ 58  32]]


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# Save treats-task TF-IDF baselines (vectoriser + 3 classifiers)
save_sklearn(vec_treats,        "treats_tfidf_vectorizer")
save_sklearn(clf_treats_logreg, "treats_tfidf_logreg")
save_sklearn(clf_treats_svm,    "treats_tfidf_svm")
save_sklearn(clf_treats_lgbm,   "treats_tfidf_lgbm")

## 9. Baseline 2: frozen BioBERT embeddings (treats task)

The frozen encoder and the `extract_frozen_embeddings` function from Section 3c are reused, followed by the same three classifiers.

In [ ]:
X_train_frozen_treats = extract_frozen_embeddings(X_train_treats)
X_test_frozen_treats = extract_frozen_embeddings(X_test_treats)
print("Frozen embedding shape - train:", X_train_frozen_treats.shape, "test:", X_test_frozen_treats.shape)


Frozen embedding shape - train: (2259, 768) test: (505, 768)


In [ ]:
clf_treats_frozen_logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_treats_frozen_logreg.fit(X_train_frozen_treats, y_train_treats)
preds_treats_frozen_logreg = clf_treats_frozen_logreg.predict(X_test_frozen_treats)

print("=== TREATS BASELINE 2a (Frozen BioBERT + Logistic Regression) ===")
print(classification_report(y_test_treats, preds_treats_frozen_logreg, digits=3))
print(confusion_matrix(y_test_treats, preds_treats_frozen_logreg))


=== TREATS BASELINE 2a (Frozen BioBERT + Logistic Regression) ===
              precision    recall  f1-score   support

           0      0.924     0.824     0.871       415
           1      0.459     0.689     0.551        90

    accuracy                          0.800       505
   macro avg      0.692     0.756     0.711       505
weighted avg      0.841     0.800     0.814       505

[[342  73]
 [ 28  62]]


In [ ]:
clf_treats_frozen_svm = LinearSVC(class_weight="balanced", max_iter=5000)
clf_treats_frozen_svm.fit(X_train_frozen_treats, y_train_treats)
preds_treats_frozen_svm = clf_treats_frozen_svm.predict(X_test_frozen_treats)

print("=== TREATS BASELINE 2b (Frozen BioBERT + SVM) ===")
print(classification_report(y_test_treats, preds_treats_frozen_svm, digits=3))
print(confusion_matrix(y_test_treats, preds_treats_frozen_svm))


=== TREATS BASELINE 2b (Frozen BioBERT + SVM) ===
              precision    recall  f1-score   support

           0      0.930     0.867     0.898       415
           1      0.534     0.700     0.606        90

    accuracy                          0.838       505
   macro avg      0.732     0.784     0.752       505
weighted avg      0.860     0.838     0.846       505

[[360  55]
 [ 27  63]]


In [ ]:
clf_treats_frozen_lgbm = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    is_unbalance=True,
    random_state=42,
)
clf_treats_frozen_lgbm.fit(X_train_frozen_treats, y_train_treats)
preds_treats_frozen_lgbm = clf_treats_frozen_lgbm.predict(X_test_frozen_treats)

print("=== TREATS BASELINE 2c (Frozen BioBERT + LightGBM) ===")
print(classification_report(y_test_treats, preds_treats_frozen_lgbm, digits=3))
print(confusion_matrix(y_test_treats, preds_treats_frozen_lgbm))


[LightGBM] [Info] Number of positive: 566, number of negative: 1693
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.022692 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 195840
[LightGBM] [Info] Number of data points in the train set: 2259, number of used features: 768
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.250553 -> initscore=-1.095663
[LightGBM] [Info] Start training from score -1.095663
=== TREATS BASELINE 2c (Frozen BioBERT + LightGBM) ===
              precision    recall  f1-score   support

           0      0.856     0.945     0.898       415
           1      0.511     0.267     0.350        90

    accuracy                          0.824       505
   macro avg      0.683     0.606     0.624       505
weighted avg      0.794     0.824     0.800       505

[[392  23]
 [ 66  24]]


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# Save treats-task frozen-BioBERT classifiers
save_sklearn(clf_treats_frozen_logreg, "treats_frozen_logreg")
save_sklearn(clf_treats_frozen_svm,    "treats_frozen_svm")
save_sklearn(clf_treats_frozen_lgbm,   "treats_frozen_lgbm")

## 10. Fine-tuned BioBERT (treats task)

Same architecture, settings and class-weighted loss as Section 4, with the class weights computed from the treats training set. A separate trainer class receives the weights as an argument, so the two tasks share no state.

In [ ]:
MODEL_NAME_TREATS = "dmis-lab/biobert-base-cased-v1.1"

tokenizer_treats = AutoTokenizer.from_pretrained(MODEL_NAME_TREATS)
tokenizer_treats.add_special_tokens({"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]})

def to_hf_dataset_treats(examples):
    return Dataset.from_dict({
        "text": [e["text"] for e in examples],
        "label": [e["label"] for e in examples],
    })

train_ds_treats = to_hf_dataset_treats(treats_train_examples)
val_ds_treats   = to_hf_dataset_treats(treats_val_examples)
test_ds_treats  = to_hf_dataset_treats(treats_test_examples)

def tok_treats(batch):
    return tokenizer_treats(batch["text"], truncation=True, max_length=256)

train_ds_treats = train_ds_treats.map(tok_treats, batched=True)
val_ds_treats   = val_ds_treats.map(tok_treats, batched=True)
test_ds_treats  = test_ds_treats.map(tok_treats, batched=True)

collator_treats = DataCollatorWithPadding(tokenizer=tokenizer_treats)

# class weights for the loss: positive-class weight = n_neg / n_pos (Equation 3.9)
n_pos_treats = sum(y_train_treats)
n_neg_treats = len(y_train_treats) - n_pos_treats
class_weights_treats = torch.tensor([1.0, n_neg_treats / max(n_pos_treats, 1)], dtype=torch.float)
print("treats class weights [neg, pos]:", class_weights_treats.tolist())

model_treats = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME_TREATS, num_labels=2)
model_treats.resize_token_embeddings(len(tokenizer_treats))

class WeightedTrainerTreats(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

args_treats = TrainingArguments(
    output_dir="./biobert_treats",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer_treats = WeightedTrainerTreats(
    model=model_treats, args=args_treats,
    train_dataset=train_ds_treats, eval_dataset=val_ds_treats,
    data_collator=collator_treats, compute_metrics=compute_metrics,
    class_weights=class_weights_treats,
)

trainer_treats.train()


Map:   0%|          | 0/2259 [00:00<?, ? examples/s]

Map:   0%|          | 0/666 [00:00<?, ? examples/s]

Map:   0%|          | 0/505 [00:00<?, ? examples/s]

treats class weights [neg, pos]: [1.0, 2.991166114807129]


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.482900,0.395364,0.875375,0.706667,0.731034,0.718644
2,0.265800,0.375096,0.882883,0.713376,0.772414,0.741722
3,0.202200,0.480352,0.890390,0.764706,0.717241,0.740214


TrainOutput(global_step=426, training_loss=0.32345542549527306, metrics={'train_runtime': 55.4297, 'train_samples_per_second': 122.263, 'train_steps_per_second': 7.685, 'total_flos': 402747997994280.0, 'train_loss': 0.32345542549527306, 'epoch': 3.0})

In [ ]:
# Save the fine-tuned treats-task BioBERT (weights + config + tokenizer)
save_hf(trainer_treats, tokenizer_treats, "biobert_treats_final")

## 11. Test-set evaluation (treats task)

The 505 test examples are built from BioRED's 100 held-out test abstracts.

In [ ]:
test_results_treats = trainer_treats.evaluate(test_ds_treats)
print("=== BioBERT treats-task test results ===")
for k, v in test_results_treats.items():
    print(f"{k}: {v}")

preds_output_treats = trainer_treats.predict(test_ds_treats)
preds_treats = np.argmax(preds_output_treats.predictions, axis=-1)
print()
print(classification_report(y_test_treats, preds_treats, digits=3))
print(confusion_matrix(y_test_treats, preds_treats))


=== BioBERT treats-task test results ===
eval_loss: 0.5878733396530151
eval_accuracy: 0.8653465346534653
eval_precision: 0.6170212765957447
eval_recall: 0.6444444444444445
eval_f1: 0.6304347826086957
eval_runtime: 0.6966
eval_samples_per_second: 724.916
eval_steps_per_second: 22.968
epoch: 3.0

              precision    recall  f1-score   support

           0      0.922     0.913     0.918       415
           1      0.617     0.644     0.630        90

    accuracy                          0.865       505
   macro avg      0.770     0.779     0.774       505
weighted avg      0.868     0.865     0.866       505

[[379  36]
 [ 32  58]]


### Treats task: test-set results

| Model | Accuracy | Precision | Recall | F1 |
|---|---|---|---|---|
| TF-IDF + logistic regression | 0.800 | 0.427 | 0.356 | 0.388 |
| TF-IDF + linear SVM | 0.818 | 0.486 | 0.389 | 0.432 |
| TF-IDF + LightGBM | 0.764 | 0.344 | 0.356 | 0.350 |
| Frozen BioBERT + logistic regression | 0.800 | 0.459 | 0.689 | 0.551 |
| Frozen BioBERT + linear SVM | 0.838 | 0.534 | 0.700 | 0.606 |
| Frozen BioBERT + LightGBM | 0.824 | 0.511 | 0.267 | 0.350 |
| **Fine-tuned BioBERT** | **0.865** | **0.617** | **0.644** | **0.630** |

Precision, recall and F1 are for the positive (treats proxy) class, on 505 test examples of which 90 are positive.

**Notes.** Labels for both tasks are derived from document-level relations by sentence-level distant supervision, so some sentence labels are noisy. The treats label is the `Negative_Correlation` proxy described in Section 6.

## 12. Saved models

Everything trained above is saved under `SAVE_DIR`: for each task, the TF-IDF vectoriser and the six baseline classifiers (joblib files), and the fine-tuned BioBERT model with its tokenizer (`biobert_causes_final`, `biobert_treats_final`). They can be reloaded without retraining:

```python
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification

vec = joblib.load(f"{SAVE_DIR}/causes_tfidf_vectorizer.joblib")
clf = joblib.load(f"{SAVE_DIR}/causes_tfidf_logreg.joblib")

tok   = AutoTokenizer.from_pretrained(f"{SAVE_DIR}/biobert_causes_final")
model = AutoModelForSequenceClassification.from_pretrained(f"{SAVE_DIR}/biobert_causes_final")
```

In [ ]:
# List everything that was saved, with sizes
for root, _, files in os.walk(SAVE_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p)/1e6:8.1f} MB  {os.path.relpath(p, SAVE_DIR)}")